# Aula 3 — Segmentar, rotular, medir

**Ideia central:** Segmentar é decidir o que é objeto. Rotular é dar um número a cada objeto. Descrever é transformar cada objeto numa linha de tabela. Depois disso, é planilha.

**Ementa:** Segmentação por limiar • Otsu • Índices de vegetação • Componentes conexos • Representação e descrição

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io, color, filters, measure, morphology, segmentation
from scipy import ndimage as ndi

## Parte 1: Cobertura verde com ExG + Otsu

O índice ExG (Woebbecke et al., 1995) usa coordenadas cromáticas para destacar vegetação.

In [ ]:
# Carregue sua imagem de lavoura
# from google.colab import files
# nome = list(files.upload().keys())[0]
# img = io.imread(nome)[..., :3].astype(float)

# Exemplo sintético
from skimage import data
img = data.grass().astype(float)  # Substitua por sua imagem

# Coordenadas cromáticas
soma = img.sum(axis=2) + 1e-6
r, g, b = (img[..., i] / soma for i in range(3))

# Índice ExG
exg = 2 * g - r - b

# Limiar de Otsu
t = filters.threshold_otsu(exg)
planta = exg > t

print(f"Cobertura verde: {100 * planta.mean():.1f}%  (limiar Otsu = {t:.3f})")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(img.astype(np.uint8))
ax[0].set_title("Original")
ax[1].imshow(exg, cmap="RdYlGn")
ax[1].set_title("ExG")
ax[2].imshow(planta, cmap="Greens")
ax[2].set_title(f"Máscara (Otsu = {t:.3f})")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## Parte 2: Contar e descrever sementes

Pipeline: cinza → Otsu → abertura → rotulação → descritores

In [ ]:
# Simula uma bandeja de sementes (use sua imagem real)
# sem = io.imread("sementes.jpg")[..., :3]

# Para demonstração, usamos uma imagem de exemplo
from skimage import data
sem = data.coins()  # Substitua por imagem de sementes
cinza = sem if sem.ndim == 2 else color.rgb2gray(sem)

# Binarização
bin_ = cinza > filters.threshold_otsu(cinza)
bin_ = morphology.binary_opening(bin_, morphology.disk(1))
bin_ = morphology.remove_small_objects(bin_, min_size=25)

# Rotulação (8-vizinhança)
rot = measure.label(bin_, connectivity=2)
print(f"Objetos encontrados: {rot.max()}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cinza, cmap="gray")
ax[0].set_title("Cinza")
ax[1].imshow(bin_, cmap="gray")
ax[1].set_title("Máscara binária")
ax[2].imshow(rot, cmap="nipy_spectral")
ax[2].set_title(f"Rótulos ({rot.max()} objetos)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## Parte 3: Tabela de descritores

In [ ]:
# Extração de propriedades
props = measure.regionprops_table(
    rot,
    intensity_image=cinza,
    properties=["label", "area", "perimeter", "eccentricity", "intensity_mean"]
)
df = pd.DataFrame(props)

# Circularidade
df["circularidade"] = 4 * np.pi * df["area"] / df["perimeter"]**2

# Estimativa de objetos grudados pela área mediana
mediana = df["area"].median()
df["n_estimado"] = np.maximum(1, np.round(df["area"] / mediana)).astype(int)

print(df.round(2).head(10))
print(f"\nTotal estimado de sementes: {df['n_estimado'].sum()}")

## Desafio: Separar sementes encostadas com watershed

In [ ]:
# Transformada de distância + watershed
dist = ndi.distance_transform_edt(bin_)
picos = morphology.h_maxima(dist, 2)
marcadores = measure.label(picos)
separadas = segmentation.watershed(-dist, marcadores, mask=bin_)

print(f"Depois do watershed: {separadas.max()} objetos")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(dist, cmap="hot")
ax[0].set_title("Distância")
ax[1].imshow(marcadores, cmap="nipy_spectral")
ax[1].set_title("Marcadores")
ax[2].imshow(separadas, cmap="nipy_spectral")
ax[2].set_title(f"Watershed ({separadas.max()} obj)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

---

## Desafios

### Aquecimento
Implemente Otsu em NumPy a partir do histograma (sem `threshold_otsu`) e confira se o limiar bate.

### Mão na massa
Rode o pipeline de sementes numa foto real (feijão ou soja em cartolina preta). Conte à mão e calcule o erro percentual.

### Investigação
Compare três contagens (rótulos, área mediana e watershed) em bandejas com 20, 40 e 60 sementes.

In [ ]:
# Espaço para os desafios


---

## Leitura recomendada

- GONZALEZ, R. C.; WOODS, R. E. *Processamento digital de imagens*. Cap. 10 e 11.
- OTSU, N. A threshold selection method from gray-level histograms. IEEE, 1979.
- WOEBBECKE, D. M. et al. Color indices for weed identification. ASAE, 1995.